# 🤖 Logistics AI Models — Predictive Analytics
**IBM Watsonx · Scikit-learn · Live Supabase Data**

---
| Model | Cells | Goal |
|-------|-------|------|
| **Model 1** | 2A + 2B | Predict if a delivery will be **late or on-time** |
| **Model 2** | 3A + 3B | Predict **fuel cost** for any fuel stop |
| **Model 3** | 4A + 4B | Predict if maintenance will be **planned or emergency** |

### ✅ Each cell pair is fully self-contained
You can run any model independently — no need to run previous cells first (except Cell 1 for setup).

In [1]:
# CELL 1 — Install libraries (run once)
import subprocess
pkgs = ["scikit-learn","pandas","numpy","plotly","requests","nbformat","ipython"]
for pkg in pkgs:
    subprocess.run(["pip","install","--upgrade",pkg], capture_output=True)

print("✅ All libraries installed!")
print()
print("Now run any model cell pair:")
print("  Cells 2A+2B → Late Delivery Predictor")
print("  Cells 3A+3B → Fuel Cost Predictor")
print("  Cells 4A+4B → Maintenance Risk Predictor")

✅ All libraries installed!

Now run any model cell pair:
  Cells 2A+2B → Late Delivery Predictor
  Cells 3A+3B → Fuel Cost Predictor
  Cells 4A+4B → Maintenance Risk Predictor


---
## Model 1 — Late Delivery Predictor 🚦
**Can we predict late deliveries BEFORE the truck departs?**
Run **Cell 2A** then **Cell 2B**.

In [4]:
# CELL 2A — Load data + Train Late Delivery model
import requests, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, roc_auc_score, roc_curve,
                             confusion_matrix, classification_report)
warnings.filterwarnings("ignore")

API = "https://YOUR_PROJECT_REF.supabase.co/functions/v1/analytics"
C = dict(yellow="#f0c040",green="#3de8a0",blue="#5b8cff",red="#ff5b5b",
         orange="#ff9f40",purple="#c77dff",surface="#111318",border="#1e2330")
LAYOUT = dict(paper_bgcolor=C["surface"],plot_bgcolor=C["surface"],
              font=dict(family="DM Mono, monospace",color="#e8eaf0",size=12),
              margin=dict(l=20,r=20,t=50,b=20),
              hoverlabel=dict(bgcolor="#1e2330",font_color="#e8eaf0"))

# ── Fetch data ──
print("Fetching delivery data (30–60 seconds)...")
r = requests.get(API, params={"query":"ml_late_delivery"}, timeout=90)
r.raise_for_status()
df_raw = pd.DataFrame(r.json().get("data") or [])

num_cols = ["actual_distance_miles","average_mpg","actual_duration_hours",
            "idle_time_hours","day_of_week","month","typical_distance_miles",
            "weight_lbs","pieces","years_experience","detention_minutes","late"]
for c in num_cols:
    df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce")

le_lt = LabelEncoder(); le_bt = LabelEncoder()
df_raw["load_type_enc"]    = le_lt.fit_transform(df_raw["load_type"].astype(str))
df_raw["booking_type_enc"] = le_bt.fit_transform(df_raw["booking_type"].astype(str))

df_late = df_raw.dropna(subset=["actual_distance_miles","average_mpg",
                                 "weight_lbs","years_experience","late"]).copy()

late_n   = int(df_late["late"].sum())
ontime_n = len(df_late) - late_n
print(f"Records: {len(df_late):,}  |  On-time: {ontime_n:,}  |  Late: {late_n:,}")

# ── Train ──
M1_FEATURES = ["actual_distance_miles","average_mpg","actual_duration_hours",
               "idle_time_hours","day_of_week","month","typical_distance_miles",
               "weight_lbs","years_experience","load_type_enc","booking_type_enc"]

X = df_late[M1_FEATURES].fillna(df_late[M1_FEATURES].median())
y = df_late["late"].astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {len(X_tr):,}  |  Test set: {len(X_te):,}")
print("Training Random Forest...")

m1 = RandomForestClassifier(n_estimators=100, max_depth=8,
                             class_weight="balanced", random_state=42, n_jobs=-1)
m1.fit(X_tr, y_tr)

y_pred = m1.predict(X_te)
y_prob = m1.predict_proba(X_te)[:,1]
m1_acc = accuracy_score(y_te, y_pred)
m1_auc = roc_auc_score(y_te, y_prob)

print(f"\n✅ Model 1 trained!")
print(f"  Accuracy : {m1_acc*100:.1f}%")
print(f"  ROC-AUC  : {m1_auc:.3f}  (1.0=perfect, 0.5=random)")
print()
print(classification_report(y_te, y_pred, target_names=["On-Time","Late"]))

# ── Class balance chart ──
fig = go.Figure(go.Bar(
    x=["On-Time","Late"], y=[ontime_n,late_n],
    marker_color=[C["green"],C["red"]],
    text=[f"{ontime_n:,}",f"{late_n:,}"], textposition="outside"
))
fig.update_layout(**LAYOUT,title="Training Data — Delivery Outcome Balance",
                  height=300,showlegend=False)
fig.update_yaxes(gridcolor=C["border"])
fig.update_xaxes(gridcolor="rgba(0,0,0,0)")
fig.show()

# Store for Cell 2B
m1_vars = {"m1":m1,"X_te":X_te,"y_te":y_te,"y_pred":y_pred,
           "y_prob":y_prob,"m1_acc":m1_acc,"m1_auc":m1_auc,
           "M1_FEATURES":M1_FEATURES}
print("\n→ Now run Cell 2B for charts and predictions")

Fetching delivery data (30–60 seconds)...
Records: 30,000  |  On-time: 13,420  |  Late: 16,580
Training set: 24,000  |  Test set: 6,000
Training Random Forest...

✅ Model 1 trained!
  Accuracy : 51.5%
  ROC-AUC  : 0.518  (1.0=perfect, 0.5=random)

              precision    recall  f1-score   support

     On-Time       0.46      0.50      0.48      2684
        Late       0.57      0.53      0.55      3316

    accuracy                           0.51      6000
   macro avg       0.51      0.51      0.51      6000
weighted avg       0.52      0.51      0.52      6000




→ Now run Cell 2B for charts and predictions


In [5]:
# CELL 2B — Evaluate + Predict (requires Cell 2A to have run first)
from sklearn.metrics import roc_curve, confusion_matrix
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")

C = dict(yellow="#f0c040",green="#3de8a0",blue="#5b8cff",red="#ff5b5b",
         orange="#ff9f40",purple="#c77dff",surface="#111318",border="#1e2330")
LAYOUT = dict(paper_bgcolor=C["surface"],plot_bgcolor=C["surface"],
              font=dict(family="DM Mono, monospace",color="#e8eaf0",size=12),
              margin=dict(l=20,r=20,t=50,b=20),
              hoverlabel=dict(bgcolor="#1e2330",font_color="#e8eaf0"))

# Pull variables from Cell 2A
m1       = m1_vars["m1"]
X_te     = m1_vars["X_te"]
y_te     = m1_vars["y_te"]
y_pred   = m1_vars["y_pred"]
y_prob   = m1_vars["y_prob"]
m1_auc   = m1_vars["m1_auc"]
M1_FEATS = m1_vars["M1_FEATURES"]

# ── Evaluation charts ──
importances = pd.DataFrame({"feature":M1_FEATS,
    "importance":m1.feature_importances_}).sort_values("importance",ascending=True)
fpr,tpr,_ = roc_curve(y_te, y_prob)
cm = confusion_matrix(y_te, y_pred)

fig = make_subplots(rows=1,cols=3,
    subplot_titles=("Feature Importance — What drives late deliveries?",
                    f"ROC Curve (AUC={m1_auc:.3f})","Confusion Matrix"),
    horizontal_spacing=0.1)

fig.add_trace(go.Bar(y=importances["feature"],x=importances["importance"],
    orientation="h",marker_color=C["blue"],
    hovertemplate="%{y}: %{x:.3f}<extra></extra>"),row=1,col=1)

fig.add_trace(go.Scatter(x=fpr,y=tpr,mode="lines",
    line=dict(color=C["yellow"],width=2.5),
    hovertemplate="FPR:%{x:.2f} TPR:%{y:.2f}<extra></extra>"),row=1,col=2)
fig.add_trace(go.Scatter(x=[0,1],y=[0,1],mode="lines",
    line=dict(color=C["border"],dash="dash",width=1),showlegend=False),row=1,col=2)

fig.add_trace(go.Heatmap(z=cm,x=["Pred On-Time","Pred Late"],
    y=["Actual On-Time","Actual Late"],
    colorscale=[[0,C["surface"]],[1,C["blue"]]],
    text=cm,texttemplate="%{text:,}",showscale=False),row=1,col=3)

fig.update_layout(**LAYOUT,height=380,title="Model 1 — Evaluation Charts")
fig.update_xaxes(gridcolor=C["border"])
fig.update_yaxes(gridcolor="rgba(0,0,0,0)",row=1,col=1)
fig.show()

# ── Predictions ──
print("=" * 58)
print("  LATE DELIVERY RISK PREDICTOR")
print("=" * 58)

scenarios = [
    {"name":"Short trip, experienced driver, weekday morning",
     "actual_distance_miles":300,"average_mpg":7.2,"actual_duration_hours":6,
     "idle_time_hours":1,"day_of_week":2,"month":6,"typical_distance_miles":290,
     "weight_lbs":18000,"years_experience":10,"load_type_enc":0,"booking_type_enc":0},
    {"name":"Long haul, new driver, heavy load, Friday",
     "actual_distance_miles":1800,"average_mpg":5.8,"actual_duration_hours":34,
     "idle_time_hours":9,"day_of_week":4,"month":12,"typical_distance_miles":1750,
     "weight_lbs":42000,"years_experience":1,"load_type_enc":1,"booking_type_enc":1},
    {"name":"Medium route, average driver, midweek",
     "actual_distance_miles":800,"average_mpg":6.5,"actual_duration_hours":14,
     "idle_time_hours":4,"day_of_week":1,"month":4,"typical_distance_miles":820,
     "weight_lbs":28000,"years_experience":5,"load_type_enc":0,"booking_type_enc":0},
]
for s in scenarios:
    name = s.pop("name")
    prob = m1.predict_proba(pd.DataFrame([s]))[0][1]
    risk = "🔴 HIGH RISK" if prob>0.6 else "🟡 MEDIUM RISK" if prob>0.35 else "🟢 LOW RISK"
    print(f"\n  {name}")
    print(f"  Late probability: {prob*100:.1f}%  {risk}")
print("\nTip: Change values above to predict any trip!")

  LATE DELIVERY RISK PREDICTOR

  Short trip, experienced driver, weekday morning
  Late probability: 50.8%  🟡 MEDIUM RISK

  Long haul, new driver, heavy load, Friday
  Late probability: 49.5%  🟡 MEDIUM RISK

  Medium route, average driver, midweek
  Late probability: 49.9%  🟡 MEDIUM RISK

Tip: Change values above to predict any trip!


---
## Model 2 — Fuel Cost Predictor ⛽
**Predict fuel cost for any stop given gallons, price, and truck details.**
Run **Cell 3A** then **Cell 3B**.

In [6]:
# CELL 3A — Load data + Train Fuel Cost model
import requests, warnings
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
warnings.filterwarnings("ignore")

API = "https://YOUR_PROJECT_REF.supabase.co/functions/v1/analytics"
C = dict(yellow="#f0c040",green="#3de8a0",blue="#5b8cff",red="#ff5b5b",
         orange="#ff9f40",purple="#c77dff",surface="#111318",border="#1e2330")
LAYOUT = dict(paper_bgcolor=C["surface"],plot_bgcolor=C["surface"],
              font=dict(family="DM Mono, monospace",color="#e8eaf0",size=12),
              margin=dict(l=20,r=20,t=50,b=20),
              hoverlabel=dict(bgcolor="#1e2330",font_color="#e8eaf0"))

# ── Fetch data ──
print("Fetching fuel data...")
r = requests.get(API, params={"query":"ml_fuel_cost"}, timeout=90)
r.raise_for_status()
df_fuel = pd.DataFrame(r.json().get("data") or [])

for c in ["gallons","price_per_gallon","total_cost","month","day_of_week",
          "model_year","tank_capacity_gallons"]:
    df_fuel[c] = pd.to_numeric(df_fuel[c], errors="coerce")

df_fuel = df_fuel.dropna(subset=["total_cost","gallons","price_per_gallon"])
df_fuel = df_fuel[df_fuel["total_cost"].between(10,5000)]
df_fuel = df_fuel[df_fuel["gallons"] > 0]

le_ft = LabelEncoder()
df_fuel["fuel_type_enc"] = le_ft.fit_transform(df_fuel["fuel_type"].astype(str))

print(f"Fuel records: {len(df_fuel):,}")
print(f"Avg cost per stop: ${df_fuel['total_cost'].mean():,.2f}")
print(f"Range: ${df_fuel['total_cost'].min():,.2f} – ${df_fuel['total_cost'].max():,.2f}")

# ── Train ──
M2_FEATURES = ["gallons","price_per_gallon","month","day_of_week",
               "fuel_type_enc","model_year","tank_capacity_gallons"]
avail_f = [f for f in M2_FEATURES if f in df_fuel.columns]

Xf = df_fuel[avail_f].fillna(df_fuel[avail_f].median())
yf = df_fuel["total_cost"]

Xf_tr,Xf_te,yf_tr,yf_te = train_test_split(Xf,yf,test_size=0.2,random_state=42)
print(f"Training: {len(Xf_tr):,}  |  Test: {len(Xf_te):,}")
print("Training Gradient Boosting model...")

m2 = GradientBoostingRegressor(n_estimators=100,max_depth=5,learning_rate=0.1,random_state=42)
m2.fit(Xf_tr,yf_tr)

yf_pred = m2.predict(Xf_te)
m2_mae  = mean_absolute_error(yf_te,yf_pred)
m2_r2   = r2_score(yf_te,yf_pred)
m2_rmse = float(np.sqrt(mean_squared_error(yf_te,yf_pred)))

print(f"\n✅ Model 2 trained!")
print(f"  MAE  (avg error):     ${m2_mae:,.2f}")
print(f"  RMSE (typical error): ${m2_rmse:,.2f}")
print(f"  R²   (fit quality):   {m2_r2:.3f}  (1.0=perfect)")

# ── Distribution chart ──
fig = px.histogram(df_fuel,x="total_cost",nbins=60,color_discrete_sequence=[C["orange"]])
fig.update_layout(**LAYOUT,title="Fuel Cost Distribution — Training Data",
                  height=300,xaxis_title="Total Cost ($)",yaxis_title="Count")
fig.update_xaxes(gridcolor="rgba(0,0,0,0)")
fig.update_yaxes(gridcolor=C["border"])
fig.show()

# Store for Cell 3B
m2_vars = {"m2":m2,"avail_f":avail_f,"Xf_te":Xf_te,"yf_te":yf_te,
           "yf_pred":yf_pred,"m2_mae":m2_mae,"m2_r2":m2_r2}
print("\n→ Now run Cell 3B for charts and predictions")

Fetching fuel data...
Fuel records: 30,000
Avg cost per stop: $485.88
Range: $159.52 – $992.31
Training: 24,000  |  Test: 6,000
Training Gradient Boosting model...

✅ Model 2 trained!
  MAE  (avg error):     $1.80
  RMSE (typical error): $2.31
  R²   (fit quality):   1.000  (1.0=perfect)



→ Now run Cell 3B for charts and predictions


In [7]:
# CELL 3B — Evaluate + Predict (requires Cell 3A to have run first)
import pandas as pd, numpy as np, warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings("ignore")

C = dict(yellow="#f0c040",green="#3de8a0",blue="#5b8cff",red="#ff5b5b",
         orange="#ff9f40",purple="#c77dff",surface="#111318",border="#1e2330")
LAYOUT = dict(paper_bgcolor=C["surface"],plot_bgcolor=C["surface"],
              font=dict(family="DM Mono, monospace",color="#e8eaf0",size=12),
              margin=dict(l=20,r=20,t=50,b=20),
              hoverlabel=dict(bgcolor="#1e2330",font_color="#e8eaf0"))

m2      = m2_vars["m2"]
avail_f = m2_vars["avail_f"]
Xf_te   = m2_vars["Xf_te"]
yf_te   = m2_vars["yf_te"]
yf_pred = m2_vars["yf_pred"]
m2_mae  = m2_vars["m2_mae"]
m2_r2   = m2_vars["m2_r2"]

# ── Evaluation charts ──
fi_fuel = pd.DataFrame({"feature":avail_f,
    "importance":m2.feature_importances_}).sort_values("importance",ascending=True)

idx = np.random.choice(len(yf_te), min(3000,len(yf_te)), replace=False)
act_s = np.array(yf_te)[idx]; pred_s = yf_pred[idx]

fig = make_subplots(rows=1,cols=2,
    subplot_titles=("Actual vs Predicted Fuel Cost","Feature Importance"),
    horizontal_spacing=0.12)

fig.add_trace(go.Scatter(x=act_s,y=pred_s,mode="markers",
    marker=dict(color=C["orange"],size=3,opacity=0.5),
    hovertemplate="Actual:$%{x:,.0f}<br>Pred:$%{y:,.0f}<extra></extra>"),row=1,col=1)
mn,mx=float(min(act_s)),float(max(act_s))
fig.add_trace(go.Scatter(x=[mn,mx],y=[mn,mx],mode="lines",
    line=dict(color=C["yellow"],dash="dash",width=1.5),showlegend=False),row=1,col=1)

fig.add_trace(go.Bar(y=fi_fuel["feature"],x=fi_fuel["importance"],
    orientation="h",marker_color=C["orange"],
    hovertemplate="%{y}:%{x:.3f}<extra></extra>"),row=1,col=2)

fig.update_layout(**LAYOUT,height=360,
    title=f"Model 2 — Fuel Cost  R²={m2_r2:.3f}  MAE=${m2_mae:,.2f}")
fig.update_xaxes(tickprefix="$",row=1,col=1,gridcolor=C["border"])
fig.update_yaxes(tickprefix="$",row=1,col=1,gridcolor=C["border"])
fig.update_xaxes(gridcolor=C["border"],row=1,col=2)
fig.update_yaxes(gridcolor="rgba(0,0,0,0)",row=1,col=2)
fig.show()

# ── Predictions ──
print("=" * 55)
print("  FUEL COST PREDICTOR")
print("=" * 55)

scenarios_f = [
    {"name":"Short refuel — 80 gallons, new truck",
     "gallons":80,"price_per_gallon":3.85,"month":6,"day_of_week":2,
     "fuel_type_enc":0,"model_year":2022,"tank_capacity_gallons":200},
    {"name":"Full tank — 200 gallons, long haul winter",
     "gallons":200,"price_per_gallon":4.20,"month":12,"day_of_week":4,
     "fuel_type_enc":0,"model_year":2018,"tank_capacity_gallons":250},
    {"name":"Older truck — 150 gallons, high mileage",
     "gallons":150,"price_per_gallon":4.10,"month":3,"day_of_week":1,
     "fuel_type_enc":0,"model_year":2015,"tank_capacity_gallons":150},
]
for s in scenarios_f:
    name = s.pop("name")
    X_new = pd.DataFrame([{k:s[k] for k in avail_f}])
    pred = m2.predict(X_new)[0]
    simple = s["gallons"] * s["price_per_gallon"]
    print(f"\n  {name}")
    print(f"  Simple (gallons × price): ${simple:,.2f}")
    print(f"  Model prediction:         ${pred:,.2f}")
print("\nTip: Change gallons or price_per_gallon to forecast any stop!")

  FUEL COST PREDICTOR

  Short refuel — 80 gallons, new truck
  Simple (gallons × price): $308.00
  Model prediction:         $304.25

  Full tank — 200 gallons, long haul winter
  Simple (gallons × price): $840.00
  Model prediction:         $839.59

  Older truck — 150 gallons, high mileage
  Simple (gallons × price): $615.00
  Model prediction:         $614.34

Tip: Change gallons or price_per_gallon to forecast any stop!


---
## Model 3 — Unplanned Maintenance Predictor 🔧
**Will the next maintenance event be planned or an emergency breakdown?**
Run **Cell 4A** then **Cell 4B**.

In [2]:
# CELL 4A — Load data + Train Maintenance Risk model
import requests, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, roc_auc_score, classification_report)
warnings.filterwarnings("ignore")

API = "https://YOUR_PROJECT_REF.supabase.co/functions/v1/analytics"
C = dict(yellow="#f0c040",green="#3de8a0",blue="#5b8cff",red="#ff5b5b",
         orange="#ff9f40",purple="#c77dff",surface="#111318",border="#1e2330")
LAYOUT = dict(paper_bgcolor=C["surface"],plot_bgcolor=C["surface"],
              font=dict(family="DM Mono, monospace",color="#e8eaf0",size=12),
              margin=dict(l=20,r=20,t=50,b=20),
              hoverlabel=dict(bgcolor="#1e2330",font_color="#e8eaf0"))

# ── Fetch data ──
print("Fetching maintenance data...")
r = requests.get(API, params={"query":"ml_maintenance"}, timeout=90)
r.raise_for_status()
df_m = pd.DataFrame(r.json().get("data") or [])

for c in ["model_year","total_cost","downtime_hours","odometer_reading",
          "labor_hours","days_since_last_service","prior_maintenance_count","is_unplanned"]:
    df_m[c] = pd.to_numeric(df_m[c], errors="coerce")

df_m = df_m.dropna(subset=["total_cost","is_unplanned"])
df_m = df_m[df_m["total_cost"] > 0]

# is_unplanned = 1 for Repair, Engine, Transmission, Brake (reactive)
# is_unplanned = 0 for Inspection, Preventive, Tire (scheduled)
le_mt = LabelEncoder()
df_m["maint_type_enc"] = le_mt.fit_transform(df_m["maintenance_type"].astype(str))

# Print label mapping so we know which encoded values to use in predictions
print("Maintenance type encoding:")
for i, cls in enumerate(le_mt.classes_):
    unplanned = "🔴 REACTIVE" if cls in ["Repair","Engine","Transmission","Brake"] else "🟢 ROUTINE"
    print(f"  {i} = {cls}  {unplanned}")

unplanned = int(df_m["is_unplanned"].sum())
planned   = len(df_m) - unplanned
up_avg = float(df_m[df_m["is_unplanned"]==1]["total_cost"].mean())
pl_avg = float(df_m[df_m["is_unplanned"]==0]["total_cost"].mean())

print(f"\nRecords: {len(df_m):,}  |  Routine: {planned}  |  Reactive: {unplanned}")
print(f"Avg routine cost:  ${pl_avg:,.2f}")
print(f"Avg reactive cost: ${up_avg:,.2f}  ({up_avg/pl_avg:.1f}x more expensive!)")

# ── Cost comparison chart ──
fig = go.Figure(go.Bar(
    x=["Routine\n(Inspection/Preventive/Tire)","Reactive\n(Repair/Engine/Transmission/Brake)"],
    y=[pl_avg,up_avg],
    marker_color=[C["green"],C["red"]],
    text=[f"${pl_avg:,.0f}",f"${up_avg:,.0f}"],textposition="outside"
))
fig.update_layout(**LAYOUT,title="Average Cost: Routine vs Reactive Maintenance",
    height=320,showlegend=False)
fig.update_yaxes(tickprefix="$",gridcolor=C["border"])
fig.update_xaxes(gridcolor="rgba(0,0,0,0)")
fig.show()

# ── Train ──
M3_FEATURES = ["model_year","odometer_reading","days_since_last_service",
               "prior_maintenance_count","maint_type_enc","downtime_hours","labor_hours"]
avail_m = [f for f in M3_FEATURES if f in df_m.columns]

Xm = df_m[avail_m].fillna(df_m[avail_m].median())
ym = df_m["is_unplanned"].astype(int)

Xm_tr,Xm_te,ym_tr,ym_te = train_test_split(Xm,ym,test_size=0.2,random_state=42,stratify=ym)
print(f"\nTraining: {len(Xm_tr):,}  |  Test: {len(Xm_te):,}")
print("Training Random Forest...")

m3 = RandomForestClassifier(n_estimators=150,max_depth=6,
                             class_weight="balanced",random_state=42,n_jobs=-1)
m3.fit(Xm_tr,ym_tr)

ym_pred = m3.predict(Xm_te)
ym_prob = m3.predict_proba(Xm_te)[:,1]
m3_acc = accuracy_score(ym_te,ym_pred)
m3_auc = roc_auc_score(ym_te,ym_prob)

print(f"\n✅ Model 3 trained!")
print(f"  Accuracy : {m3_acc*100:.1f}%")
print(f"  ROC-AUC  : {m3_auc:.3f}")
print()
print(classification_report(ym_te,ym_pred,target_names=["Routine","Reactive"]))

# Store for Cell 4B
m3_vars = {"m3":m3,"avail_m":avail_m,"Xm_te":Xm_te,"ym_te":ym_te,
           "ym_pred":ym_pred,"ym_prob":ym_prob,"m3_auc":m3_auc,
           "le_mt":le_mt}
print("\n→ Now run Cell 4B for charts and predictions")

Fetching maintenance data...
Maintenance type encoding:
  0 = Brake  🔴 REACTIVE
  1 = Engine  🔴 REACTIVE
  2 = Inspection  🟢 ROUTINE
  3 = Preventive  🟢 ROUTINE
  4 = Repair  🔴 REACTIVE
  5 = Tire  🟢 ROUTINE
  6 = Transmission  🔴 REACTIVE

Records: 2,920  |  Routine: 1268  |  Reactive: 1652
Avg routine cost:  $1,658.81
Avg reactive cost: $2,195.64  (1.3x more expensive!)



Training: 2,336  |  Test: 584
Training Random Forest...

✅ Model 3 trained!
  Accuracy : 95.2%
  ROC-AUC  : 0.993

              precision    recall  f1-score   support

     Routine       0.93      0.96      0.95       254
    Reactive       0.97      0.95      0.96       330

    accuracy                           0.95       584
   macro avg       0.95      0.95      0.95       584
weighted avg       0.95      0.95      0.95       584


→ Now run Cell 4B for charts and predictions


In [3]:
# CELL 4B — Evaluate + Predict (requires Cell 4A to have run first)
import pandas as pd, numpy as np, warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import roc_curve, confusion_matrix
warnings.filterwarnings("ignore")

C = dict(yellow="#f0c040",green="#3de8a0",blue="#5b8cff",red="#ff5b5b",
         orange="#ff9f40",purple="#c77dff",surface="#111318",border="#1e2330")
LAYOUT = dict(paper_bgcolor=C["surface"],plot_bgcolor=C["surface"],
              font=dict(family="DM Mono, monospace",color="#e8eaf0",size=12),
              margin=dict(l=20,r=20,t=50,b=20),
              hoverlabel=dict(bgcolor="#1e2330",font_color="#e8eaf0"))

m3      = m3_vars["m3"]
avail_m = m3_vars["avail_m"]
Xm_te   = m3_vars["Xm_te"]
ym_te   = m3_vars["ym_te"]
ym_pred = m3_vars["ym_pred"]
ym_prob = m3_vars["ym_prob"]
m3_auc  = m3_vars["m3_auc"]
le_mt   = m3_vars["le_mt"]

# Get the encoded value for each type dynamically
type_enc = {cls: i for i, cls in enumerate(le_mt.classes_)}
print("Using these encodings for predictions:")
for t, e in type_enc.items():
    print(f"  maint_type_enc={e}  →  {t}")

# ── Evaluation charts ──
fi_m = pd.DataFrame({"feature":avail_m,
    "importance":m3.feature_importances_}).sort_values("importance",ascending=True)
fpr_m,tpr_m,_ = roc_curve(ym_te,ym_prob)
cm_m = confusion_matrix(ym_te,ym_pred)

fig = make_subplots(rows=1,cols=3,
    subplot_titles=("Feature Importance",f"ROC Curve (AUC={m3_auc:.3f})",
                    "Confusion Matrix\nRoutine vs Reactive"),
    horizontal_spacing=0.1)

fig.add_trace(go.Bar(y=fi_m["feature"],x=fi_m["importance"],
    orientation="h",marker_color=C["purple"],
    hovertemplate="%{y}:%{x:.3f}<extra></extra>"),row=1,col=1)

fig.add_trace(go.Scatter(x=fpr_m,y=tpr_m,mode="lines",
    line=dict(color=C["yellow"],width=2.5),name=f"AUC={m3_auc:.3f}"),row=1,col=2)
fig.add_trace(go.Scatter(x=[0,1],y=[0,1],mode="lines",
    line=dict(color=C["border"],dash="dash",width=1),showlegend=False),row=1,col=2)

fig.add_trace(go.Heatmap(z=cm_m,x=["Pred Routine","Pred Reactive"],
    y=["Actual Routine","Actual Reactive"],
    colorscale=[[0,C["surface"]],[1,C["purple"]]],
    text=cm_m,texttemplate="%{text:,}",showscale=False),row=1,col=3)

fig.update_layout(**LAYOUT,height=380,
    title=f"Model 3 — Reactive Maintenance Risk — AUC={m3_auc:.3f}")
fig.update_xaxes(gridcolor=C["border"])
fig.update_yaxes(gridcolor="rgba(0,0,0,0)",row=1,col=1)
fig.show()

# ── Predictions ──
print("=" * 60)
print("  REACTIVE MAINTENANCE RISK PREDICTOR")
print("  (1=Reactive: Repair/Engine/Transmission/Brake)")
print("  (0=Routine:  Inspection/Preventive/Tire)")
print("=" * 60)

trucks = [
    {"name":"New truck, routine inspection due",
     "model_year":2023,"odometer_reading":45000,"days_since_last_service":30,
     "prior_maintenance_count":3,"maint_type_enc":type_enc.get("Inspection",2),
     "downtime_hours":2,"labor_hours":1.5},
    {"name":"Old high-mileage truck, overdue service",
     "model_year":2016,"odometer_reading":520000,"days_since_last_service":180,
     "prior_maintenance_count":28,"maint_type_enc":type_enc.get("Repair",4),
     "downtime_hours":0,"labor_hours":3},
    {"name":"Mid-age truck, regular preventive schedule",
     "model_year":2019,"odometer_reading":210000,"days_since_last_service":45,
     "prior_maintenance_count":14,"maint_type_enc":type_enc.get("Preventive",3),
     "downtime_hours":0,"labor_hours":2},
    {"name":"Any truck — transmission check",
     "model_year":2018,"odometer_reading":300000,"days_since_last_service":90,
     "prior_maintenance_count":18,"maint_type_enc":type_enc.get("Transmission",5),
     "downtime_hours":0,"labor_hours":4},
]
for t in trucks:
    name = t.pop("name")
    X_new = pd.DataFrame([{k:t[k] for k in avail_m if k in t}])
    prob_up = m3.predict_proba(X_new)[0][1]
    risk = "🔴 HIGH RISK" if prob_up>0.55 else "🟡 MEDIUM RISK" if prob_up>0.3 else "🟢 LOW RISK"
    print(f"\n  {name}")
    print(f"  Reactive probability: {prob_up*100:.1f}%  {risk}")
print("\nTip: Use type_enc dict above to get the right maint_type_enc value for any type!")

Using these encodings for predictions:
  maint_type_enc=0  →  Brake
  maint_type_enc=1  →  Engine
  maint_type_enc=2  →  Inspection
  maint_type_enc=3  →  Preventive
  maint_type_enc=4  →  Repair
  maint_type_enc=5  →  Tire
  maint_type_enc=6  →  Transmission


  REACTIVE MAINTENANCE RISK PREDICTOR
  (1=Reactive: Repair/Engine/Transmission/Brake)
  (0=Routine:  Inspection/Preventive/Tire)

  New truck, routine inspection due
  Reactive probability: 25.5%  🟢 LOW RISK

  Old high-mileage truck, overdue service
  Reactive probability: 50.3%  🟡 MEDIUM RISK

  Mid-age truck, regular preventive schedule
  Reactive probability: 12.1%  🟢 LOW RISK

  Any truck — transmission check
  Reactive probability: 56.0%  🔴 HIGH RISK

Tip: Use type_enc dict above to get the right maint_type_enc value for any type!


---
## ✅ All 3 Models Complete

| Model | What it predicts | How to use operationally |
|-------|-----------------|--------------------------|
| **Late Delivery** | Late arrival risk % | Flag high-risk trips before dispatch |
| **Fuel Cost** | Predicted stop cost ($) | Budget fuel spend per route per month |
| **Maintenance Risk** | Emergency breakdown % | Schedule proactive service for at-risk trucks |